# Chapter 11: Pattern Matching. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/11-pattern.R`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/11-pattern.R).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/11-pattern.ipynb)

In [ ]:
# Packages that the exercises of this notebook use; the ones that are missing are installed
pkgs <- c("mgcv", "nnet")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new) > 0) install.packages(new, repos = "https://cloud.r-project.org")

## Exercise 11.6: Composition versus Addition: Why Depth Captures Interactions

The chapter's *Complex Functions* section contrasts the composition structure of deep learning, $f = f_L(\cdots f_1(x))$ with $f_i(x) = \sigma(W_i x + b_i)$, against the addition structure of the traditional models in Table 11.1, such as the generalized additive model $y = \beta_0 + f_1(x_1) + \cdots + f_k(x_k)$. This exercise makes the practical difference concrete on a purely interactive target.

a) Simulate $n = 400$ observations with two independent predictors $x_1, x_2 \sim \text{Uniform}(-1,1)$ and a purely interactive response
$$
y = 4 x_1 x_2 + \varepsilon, \qquad \varepsilon \sim N(0, 0.3^2).
$$
Note that $\mathrm{E}\left(y \mid x_1\right) = 4 x_1 \mathrm{E}\left(x_2\right) = 0$ and likewise $\mathrm{E}\left(y \mid x_2\right) = 0$: neither predictor has any marginal effect on its own, the entire signal is in how $x_1$ and $x_2$ act together. Confirm that the sample correlations $\mathrm{Cor}\left(y, x_1\right)$ and $\mathrm{Cor}\left(y, x_2\right)$ are both near zero, then split the data into a 70% training set and a 30% test set (use `set.seed(11)`).

b) Fit the additive model $\hat y = f_1(x_1) + f_2(x_2)$ with `mgcv::gam(y ~ s(x1) + s(x2))`, mirroring the GAM row of Table 11.1. Report the test MSE and compare it to the test MSE of the mean rule $\hat y = \bar y$ from the start of the chapter. Inspect `summary()` of the fit: what do the estimated degrees of freedom and $p$-values of the two smooth terms say?

c) Fit the composition $\hat y = f_2(f_1(x))$ with a small one-hidden-layer network, `nnet::nnet(y ~ x1 + x2, size = 8, linout = TRUE, decay = 0.001)`. Here the hidden layer computes $f_1(x) = \sigma(W_1 x + b_1)$ exactly as in the chapter, mixing $x_1$ and $x_2$ together before the nonlinearity. Report its test MSE on the same split and compare it to (b).

d) Explain the gap between (b) and (c) from the functional forms alone. Show that for any additive model $\hat y = f_1(x_1) + f_2(x_2)$ the mixed partial derivative $\partial^2 \hat y / (\partial x_1 \partial x_2) \equiv 0$, no matter how flexible $f_1$ and $f_2$ are individually. Since the true surface has $\partial^2 y / (\partial x_1 \partial x_2) = 4 \neq 0$, argue why this is a structural limitation of addition rather than a shortage of data or flexibility, and why composition, through the linear mixing $W_1 x$ inside $f_1$, removes it.

## Exercise 11.7: Why $R^2$ Should Be Avoided: Two Regimes

The chapter argues that $R^2$ can be arbitrarily low when a model is completely correct, and arbitrarily close to 1 when a model is completely wrong. This exercise makes both claims concrete by simulation.

a) Simulate $n = 300$ observations from a *correctly specified* linear model $y = 2 + 3x + \epsilon$, with $x \sim N(0,1)$ and noise $\epsilon \sim N(0, \sigma^2)$ using a large noise standard deviation $\sigma = 50$. Fit `lm(y ~ x)` and report $R^2$. Confirm the fitted slope is still close to the true value $\beta_1 = 3$, even though $R^2$ is tiny.

b) Simulate $n = 300$ observations from a *misspecified* relationship: $x \sim \text{Uniform}(1, 10)$ and $y = 5 + 10\sqrt{x} + \epsilon$, with $\epsilon \sim N\bigl(0, (0.4x)^2\bigr)$, so the mean function is nonlinear, and the noise variance grows with $x$. Fit `lm(y ~ x)` anyway and report $R^2$.

c) For the model in (b), compute $\mathrm{Cor}\left(|e|, x\right)$, where $e$ is the vector of residuals. What does a nonzero value tell you about an assumption of the fitted linear model, an assumption the high $R^2$ from (b) gave no hint of violating?

d) Using (a)-(c), explain in two or three sentences why $R^2$ measures neither correct specification nor predictive usefulness. Which metrics from this chapter would you report instead?

## Exercise 11.8: Comparing the Mean Rule, k-NN, and Linear Regression

This chapter introduces the mean rule and the nearest-neighbor rule as two extremes of the bias-variance spectrum, with linear regression, a parametric model richer than the mean rule, between them. Here we compare all three directly on the built-in `mtcars` data ($n = 32$), predicting `mpg` from `wt` (weight in 1000 lbs).

a) Because $n = 32$ is small, use 5-fold cross-validation rather than a single train/test split, as the chapter recommends for small samples. Write a short function that performs $k$-nearest-neighbor regression by hand: for each test point $x_0$, average the $y$-values of the $k$ training points whose $x$ is closest to $x_0$.

b) For each of the 5 folds, fit (i) the mean rule, (ii) $k$-NN with $k=3$, and (iii) linear regression on the training portion of the fold, and evaluate the mean squared error on the held-out portion. Report the average cross-validated MSE (CV-MSE) for each method.

c) Rank the three methods by CV-MSE. Relate the ranking to the chapter's description of the mean rule as "high bias, low variance" and the nearest-neighbor rule as "low bias, high variance."

d) Recompute the $k$-NN CV-MSE for $k \in \{1, 3, 5, 10\}$. Describe the pattern as $k$ grows. What value of $k$ makes $k$-NN's prediction identical to the mean rule for every test point, and why?